In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000212.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000121.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000204.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000154.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000336.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000214.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000048.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000195.jpg
/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences/uav0000117_02622_v/0000156.jpg
/

In [1]:
import os
import cv2
from pathlib import Path
from tqdm import tqdm

# CHANGE THIS to your Kaggle input path
VISDRONE_ROOT = Path("/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val")

SEQS_DIR = VISDRONE_ROOT / "sequences"
ANN_DIR = VISDRONE_ROOT / "annotations"

OUT_ROOT = Path("/kaggle/working/visdrone_person_yolo")
IMG_OUT = OUT_ROOT / "images" / "val"
LBL_OUT = OUT_ROOT / "labels" / "val"

IMG_OUT.mkdir(parents=True, exist_ok=True)
LBL_OUT.mkdir(parents=True, exist_ok=True)

print("Input:", VISDRONE_ROOT)
print("Output:", OUT_ROOT)

Input: /kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val
Output: /kaggle/working/visdrone_person_yolo


In [2]:
# Keep only person-related classes
# 1 = pedestrian, 2 = people
KEEP_CLASSES = {1: 0, 2: 0}   # map both to YOLO class 0 = person

num_images = 0
num_labels = 0

for seq_dir in tqdm(sorted(SEQS_DIR.iterdir()), desc="Sequences"):
    if not seq_dir.is_dir():
        continue

    ann_path = ANN_DIR / f"{seq_dir.name}.txt"
    if not ann_path.exists():
        print(f"Missing annotation for {seq_dir.name}")
        continue

    # frame_id -> list of boxes
    frame_boxes = {}

    with open(ann_path, "r") as f:
        for line in f:
            vals = line.strip().split(",")
            if len(vals) < 10:
                continue

            frame_id = int(vals[0])
            target_id = int(vals[1])
            x = float(vals[2])
            y = float(vals[3])
            w = float(vals[4])
            h = float(vals[5])
            score = int(vals[6])
            cls_id = int(vals[7])
            trunc = int(vals[8])
            occ = int(vals[9])

            if cls_id not in KEEP_CLASSES:
                continue
            if w <= 1 or h <= 1:
                continue

            frame_boxes.setdefault(frame_id, []).append((x, y, w, h, KEEP_CLASSES[cls_id]))

    for img_path in sorted(seq_dir.glob("*.jpg")):
        frame_id = int(img_path.stem)

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        H, W = img.shape[:2]

        # rename image to avoid duplicate filenames across sequences
        new_img_name = f"{seq_dir.name}_{img_path.name}"
        new_lbl_name = f"{seq_dir.name}_{img_path.stem}.txt"

        out_img_path = IMG_OUT / new_img_name
        out_lbl_path = LBL_OUT / new_lbl_name

        # save image
        cv2.imwrite(str(out_img_path), img)
        num_images += 1

        # save label
        boxes = frame_boxes.get(frame_id, [])
        with open(out_lbl_path, "w") as fw:
            for x, y, w, h, cls in boxes:
                xc = (x + w / 2) / W
                yc = (y + h / 2) / H
                wn = w / W
                hn = h / H
                fw.write(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")
                num_labels += 1

print(f"Done. Images: {num_images}, Labels: {num_labels}")

Sequences: 100%|██████████| 7/7 [03:09<00:00, 27.01s/it]

Done. Images: 2846, Labels: 50312


In [3]:
yaml_text = """
path: /kaggle/working/visdrone_person_yolo
train: images/val
val: images/val

nc: 1
names: ['person']
"""

with open("/kaggle/working/visdrone_person_yolo/visdrone_person.yaml", "w") as f:
    f.write(yaml_text)

print("YAML file created.")

YAML file created.


In [4]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.4 MB/s eta 0:00:00a 0:00:01


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # or yolov8n.pt for faster/lighter

model.train(
    data="/kaggle/working/visdrone_person_yolo/visdrone_person.yaml",
    epochs=30,
    imgsz=1280,
    batch=8,
    device=0,
    workers=4,
    project="/kaggle/working",
    name="visdrone_person_yolo_train"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.32 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/visdrone_person_yolo/visdrone_person.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7878889aade0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [6]:
best_model = YOLO("/kaggle/working/visdrone_person_yolo_train/weights/best.pt")
best_model.export(format="onnx", imgsz=1280, opset=12, simplify=True)

Ultralytics 8.4.32 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/kaggle/working/visdrone_person_yolo_train/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 5, 33600) (21.5 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 265ms
Prepared 2 packages in 2.53s
Installed 2 packages in 11ms
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.90

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with onnxslim 0.1.90...
ONNX: expo

'/kaggle/working/visdrone_person_yolo_train/weights/best.onnx'

In [2]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:0000:0100:01


In [3]:
import os
import cv2
import time
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from IPython.display import Video, display

# =========================================================
# CONFIG
# =========================================================
MODEL_PATH = "/kaggle/input/models/vineet77/visdrone-model/pytorch/default/1/best.pt"

# CHANGE THIS to your dataset path
VISDRONE_SEQ_ROOT = Path("/kaggle/input/datasets/vineet77/visdrone-dataset/VisDrone2019-MOT-val/sequences")

OUT_DIR = Path("/kaggle/working/inference_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 1280
CONF_THRESH = 0.50
SEQ_NAME = None   # Example: "uav0000086_00000_v"

# =========================================================
# SIMPLE BYTE TRACKER
# =========================================================
def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2]-box1[0]) * max(0, box1[3]-box1[1])
    area2 = max(0, box2[2]-box2[0]) * max(0, box2[3]-box2[1])
    union = area1 + area2 - inter + 1e-6
    return inter / union

class Track:
    def __init__(self, box, score, track_id):
        self.box = box
        self.score = score
        self.id = track_id
        self.age = 0
        self.hits = 1

class BYTETrackerSimple:
    def __init__(self, iou_thresh=0.3, max_age=30):
        self.iou_thresh = iou_thresh
        self.max_age = max_age
        self.tracks = []
        self.next_id = 1

    def update(self, detections):
        updated_tracks = []
        unmatched_dets = list(range(len(detections)))
        unmatched_trks = list(range(len(self.tracks)))

        if len(self.tracks) > 0 and len(detections) > 0:
            iou_matrix = np.zeros((len(self.tracks), len(detections)), dtype=np.float32)

            for t, trk in enumerate(self.tracks):
                for d, det in enumerate(detections):
                    iou_matrix[t, d] = iou(trk.box, det[:4])

            row_ind, col_ind = linear_sum_assignment(-iou_matrix)

            matched = []
            for r, c in zip(row_ind, col_ind):
                if iou_matrix[r, c] >= self.iou_thresh:
                    matched.append((r, c))

            unmatched_trks = [t for t in range(len(self.tracks)) if t not in [m[0] for m in matched]]
            unmatched_dets = [d for d in range(len(detections)) if d not in [m[1] for m in matched]]

            for t, d in matched:
                self.tracks[t].box = detections[d][:4]
                self.tracks[t].score = detections[d][4]
                self.tracks[t].age = 0
                self.tracks[t].hits += 1
                updated_tracks.append(self.tracks[t])

        for t in unmatched_trks:
            self.tracks[t].age += 1
            if self.tracks[t].age <= self.max_age:
                updated_tracks.append(self.tracks[t])

        for d in unmatched_dets:
            new_track = Track(detections[d][:4], detections[d][4], self.next_id)
            self.next_id += 1
            updated_tracks.append(new_track)

        self.tracks = updated_tracks

        results = []
        for trk in self.tracks:
            results.append({
                "id": trk.id,
                "bbox": trk.box,
                "score": trk.score
            })
        return results

# =========================================================
# LOAD MODEL
# =========================================================
model = YOLO(MODEL_PATH)
tracker = BYTETrackerSimple(iou_thresh=0.3, max_age=30)

# =========================================================
# PICK SEQUENCE
# =========================================================
seq_dirs = sorted([d for d in VISDRONE_SEQ_ROOT.iterdir() if d.is_dir()])

if len(seq_dirs) == 0:
    raise ValueError("No sequences found.")

if SEQ_NAME is None:
    seq_dir = seq_dirs[0]
else:
    seq_dir = VISDRONE_SEQ_ROOT / SEQ_NAME

print(f"Running inference on sequence: {seq_dir.name}")

img_paths = sorted(seq_dir.glob("*.jpg"))
if len(img_paths) == 0:
    raise ValueError(f"No images found in {seq_dir}")

# =========================================================
# VIDEO WRITER
# =========================================================
first = cv2.imread(str(img_paths[0]))
H, W = first.shape[:2]

out_video_path = OUT_DIR / f"{seq_dir.name}_tracked_fps.mp4"
writer = cv2.VideoWriter(
    str(out_video_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    20.0,
    (W, H)
)

# =========================================================
# RUN INFERENCE
# =========================================================
total_time = 0.0
total_frames = 0

for frame_idx, img_path in enumerate(img_paths, start=1):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue

    t0 = time.time()

    # ------------------ Detection ------------------
    results = model(frame, imgsz=INPUT_SIZE, conf=CONF_THRESH, verbose=False)
    detections = []

    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        clss = boxes.cls.cpu().numpy()

        for box, conf, cls in zip(xyxy, confs, clss):
            if int(cls) != 0:  # keep person only
                continue
            x1, y1, x2, y2 = box
            detections.append([x1, y1, x2, y2, conf])

    # ------------------ Tracking ------------------
    tracks = tracker.update(detections)

    # ------------------ Draw Tracks ------------------
    for trk in tracks:
        x1, y1, x2, y2 = map(int, trk["bbox"])
        tid = trk["id"]
        score = trk["score"]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"ID {tid} {score:.2f}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # ------------------ FPS Calculation ------------------
    dt = time.time() - t0
    total_time += dt
    total_frames += 1

    fps_now = 1.0 / (dt + 1e-6)
    avg_fps = total_frames / total_time

    # ------------------ Draw FPS (TOP-LEFT) ------------------
    cv2.putText(
        frame,
        f"FPS: {fps_now:.1f}",
        (20, 35),   # top-left
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"AVG FPS: {avg_fps:.1f}",
        (20, 75),   # below FPS
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 0),
        2
    )

    # Optional frame number
    cv2.putText(
        frame,
        f"Frame: {frame_idx}",
        (20, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    writer.write(frame)

writer.release()

print(f"Saved video: {out_video_path}")
print(f"Average FPS: {avg_fps:.2f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Running inference on sequence: uav0000086_00000_v
Saved video: /kaggle/working/inference_results/uav0000086_00000_v_tracked_fps.mp4
Average FPS: 19.28
